# Edge simtable fill

Fill STS / payload from the SQ-GAN v1 pass, D6 codec-quality report, SPI+STS fusion.


In [ ]:
import glob, os, subprocess, sys
subprocess.run(["git", "clone", "--depth", "1", "-q", "https://github.com/saltypal/SemanticSchedulerEnd2End.git", "/tmp/repo"], check=True)
print("repo @", subprocess.run(["git", "-C", "/tmp/repo", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "/tmp/repo"], check=False)
c2 = sorted({os.path.dirname(p) for p in glob.glob("/kaggle/input/**/cache_sqgan/shards", recursive=True)})
tab = sorted(glob.glob("/kaggle/input/**/simtable_v1.jsonl", recursive=True))
print("C2:", c2, "\nTABLE:", tab)
assert c2 and tab, "inputs not mounted"
n = len(glob.glob(c2[0] + "/shards/shard_*/done.json")); print("finished shards:", n)


In [ ]:
def run(*a):
    r = subprocess.run([sys.executable, "-m", *a], cwd="/tmp/repo", capture_output=True, text=True)
    print(r.stdout[-3000:]); print(r.stderr[-2000:])
    if r.returncode: raise SystemExit("failed: " + " ".join(a))
run("parameters.edge.simtable", "fill-codec", "--table", tab[0], "--c2", c2[0], "--out", "/kaggle/working/tab")
run("Evaluation.edge.report", "codec-quality", "--c2", c2[0], "--table", "/kaggle/working/tab/simtable_v1.jsonl",
    "--out", "/kaggle/working/quality")
run("parameters.edge.sts_fusion", "--table", "/kaggle/working/tab", "--out", "/kaggle/working/fusion")
